In [6]:
import pyreadstat
import pandas as pd
from glob import glob
import numpy as np


## Pip UI vs API

In [13]:

pip_ui_prev = pd.read_csv(
    '/data/eop/compiled_country_data/pip_interpolated_all_countries_215_at_2017_ppp_20250822.csv'
)
pip_ui_prev = pip_ui_prev[pip_ui_prev.reporting_year == 2023]
pip_ui_prev = pip_ui_prev[
    ~(pip_ui_prev.country_code == 'CHN')
    | (pip_ui_prev.reporting_level == 'national')
]

pip_ui_new = pd.read_csv(
    '/data/eop/compiled_country_data/pip_interpolated_all_countries_215_at_2017_ppp_20260401.csv'
)
pip_ui_new = pip_ui_new[pip_ui_new.reporting_year == 2023]
pip_ui_new = pip_ui_new[
    ~(pip_ui_new.country_code == 'CHN')
    | (pip_ui_new.reporting_level == 'national')
]

pip_api_old, _ = pyreadstat.read_dta(
    '/data/eop/compiled_country_data/from_pip_api/poverty_data_accessed_20260401.dta'
)

pip_api_old_2023_2017 = pip_api_old[
    (pip_api_old.year == 2023)
    & (pip_api_old.poverty_line == 2.15)
]
pip_api_old_2023_2017 = pip_api_old_2023_2017[
    ~(pip_api_old_2023_2017.country_code == 'CHN')
    | (pip_api_old_2023_2017.reporting_level == 'national')
]


pip_api_new, _ = pyreadstat.read_dta(
    '/data/eop/compiled_country_data/from_pip_api/poverty_data_accessed_20260409.dta'
)


pip_api_new_2023_2017 = pip_api_new[
    (pip_api_new.year == 2023)
    & (pip_api_new.poverty_line == 2.15)
    & (pip_api_new.ppp == 2017)
]
pip_api_new_2023_2017 = pip_api_new_2023_2017[
    ~(pip_api_new_2023_2017.country_code == 'CHN')
    | (pip_api_new_2023_2017.reporting_level == 'national')
]


In [ ]:
pip_comparison = pip_ui_new.merge(pip_api_new_2023_2017, on='country_code', suffixes=('_ui', '_api'))
pip_comparison['ratio'] = pip_comparison.headcount_api/pip_comparison.headcount_ui

In [23]:
pip_comparison.loc[
    pip_comparison.headcount_ui > 0,
    ['country_name_ui', 'headcount_ui', 'headcount_api', 'ratio']
].sort_values('ratio', ascending=False)

,country_name_ui,headcount_ui,headcount_api,ratio
217,Zimbabwe,0.38035,0.38035,1.0
0,Aruba,0.00480,0.00480,1.0
202,Ukraine,0.00030,0.00030,1.0
203,Uruguay,0.00110,0.00110,1.0
204,United States,0.01115,0.01115,1.0
...,...,...,...,...
8,American Samoa,0.00030,0.00030,1.0
9,Antigua and Barbuda,0.01430,0.01430,1.0
10,Australia,0.00730,0.00730,1.0
11,Austria,0.00515,0.00515,1.0


## Comparison with WPC

In [24]:
pd.set_option('display.max_columns', None)
aux_files = glob('/data/eop/compiled_country_data/auxiliary_data/auxiliary_data_*.csv')
latest_file = max(aux_files, key=lambda x: x.split('_')[-1].split('.')[0])
aux_data = pd.read_csv(latest_file)
print(f'Latest file: {latest_file}')

Latest file: /data/eop/compiled_country_data/auxiliary_data/auxiliary_data_20260409.csv


### Rate comparison

In [25]:
wpc = pd.read_csv(
    '/data/eop/20251209_snapshot_country_data/auxiliary_data/wdl_pov_clock_oct_2024/wdl_pov_clock_oct_2024.csv'
).rename(columns={
    'ccode': 'country_code', 'hcr_pov': 'poverty_rate_wpc'
})
wpc = wpc[wpc.year == 2023]


In [26]:
merged = wpc.merge(aux_data, on='country_code', how='outer')
merged['country'] = merged.country_x.fillna(merged.country_y)

In [27]:
for_rate_comparison = merged[['country', 'total_population_2023', 'country_code', 'poverty_rate_wpc','wb_poverty_rate_2023_povertyline_2021', 'wb_poverty_rate_2023_povertyline_2017']].copy()
wb_year = '2017' # should be 2017, made variable just for a sanity check
for_rate_comparison['rate_ratio_wb_over_wpc'] = (
    for_rate_comparison[f'wb_poverty_rate_2023_povertyline_{wb_year}'] / for_rate_comparison.poverty_rate_wpc
)
for_rate_comparison['rate_difference_wb_minus_wpc'] = (
    for_rate_comparison[f'wb_poverty_rate_2023_povertyline_{wb_year}'] - for_rate_comparison.poverty_rate_wpc
)
for_rate_comparison['poor_count_difference_wb_minus_wpc'] = for_rate_comparison.rate_difference_wb_minus_wpc * for_rate_comparison.total_population_2023
for_rate_comparison['sort_key'] = for_rate_comparison.poor_count_difference_wb_minus_wpc.abs()

for_rate_comparison['poor_count_wpc'] = for_rate_comparison.poverty_rate_wpc * for_rate_comparison.total_population_2023
for_rate_comparison['poor_count_wb'] = for_rate_comparison[f'wb_poverty_rate_2023_povertyline_{wb_year}'] * for_rate_comparison.total_population_2023
for_rate_comparison['poor_count_difference_wb_minus_wpc_checksum'] = for_rate_comparison.poor_count_wb - for_rate_comparison.poor_count_wpc

In [28]:
for_rate_comparison = (
    for_rate_comparison[
        (for_rate_comparison.rate_ratio_wb_over_wpc.notna())
        & (np.maximum(for_rate_comparison.wb_poverty_rate_2023_povertyline_2017, for_rate_comparison.poverty_rate_wpc) > 0.01)
    ].sort_values('sort_key', ascending=False)
)


In [29]:
biggest_changes = for_rate_comparison.rename(columns={
    'wb_poverty_rate_2023_povertyline_2017': 'poverty_rate_wb',
}).head(20)

In [ ]:
biggest_changes[['country' 'poverty_rate_wpc', 'poverty_rate_wb', 'poor_count_difference_wb_minus_wpc', 'rate_ratio_wb_over_wpc', 'rate_difference_wb_minus_wpc']]

,country,country_code,poverty_rate_wpc,poverty_rate_wb,poor_count_difference_wb_minus_wpc,rate_ratio_wb_over_wpc,rate_difference_wb_minus_wpc
58,Ethiopia,ETH,0.150061,0.28350,1.717250e+07,1.889233,0.133439
134,Nigeria,NGA,0.316612,0.37850,1.410312e+07,1.195468,0.061888
142,Pakistan,PAK,0.038455,0.08895,1.249770e+07,2.313083,0.050495
197,Venezuela,VEN,0.445986,0.02460,-1.192558e+07,0.055159,-0.421386
83,India,IND,0.013038,0.02070,1.101842e+07,1.587661,0.007662
159,Sudan,SDN,0.264448,0.44055,8.812660e+06,1.665926,0.176102
204,South Africa,ZAF,0.216013,0.08525,-8.265834e+06,0.394652,-0.130763
24,Brazil,BRA,0.059272,0.02665,-6.887794e+06,0.449623,-0.032622
117,Madagascar,MDG,0.674041,0.51245,-5.040976e+06,0.760266,-0.161591
203,Yemen,YEM,0.690763,0.56455,-4.971636e+06,0.817285,-0.126213


In [31]:
for_rate_comparison.poor_count_difference_wb_minus_wpc.sum() / 1e6

30.067640129706607

In [32]:
for_rate_comparison.poor_count_wb.sum() / 1e6

628.0467624183

In [33]:
for_rate_comparison.poor_count_wpc.sum() / 1e6

597.9791222885934